Schritt 2: Einführung des "Gedächtnis-Bonds" $c_k$: Wir zwingen das System, an jedem Schnitt $k$ exakt die letzten $K$ Zustände zu speichern. Dafür erfinden wir den Sammel-Index $c_k = (\alpha_{k-K+1}, \dots, \alpha_k)$. Mathematisch realisieren wir das, indem wir eine Summe über alle möglichen Zustände von $c_k$ sowie eine Indikatorfunktion $\mathbb{1}[\dots]$ einschieben. Diese Funktion wirkt wie ein strenger Filter: Sie ist nur $1$, wenn das Fenster exakt die korrekten $\alpha$'s enthält, ansonsten $0$.$$\begin{aligned} &= \sum_{\alpha_1\ldots\alpha_N} \underbrace{\sum_{c_1\dots c_N}}_{\text{Summe über Bonds}} (P_h)_{\beta\alpha_N} \left[ \prod_{k=1}^{N} W(\alpha_k, \alpha_{k-1}, \dots, \alpha_{k-K}) \cdot \mathbb{1}\big[\,c_k = (\alpha_{k-K+1}, \dots, \alpha_k)\,\big] \right] (P_h)_{\alpha_1\alpha_0} \end{aligned}$$

Stell dir vor, du bist an der Stelle $k$, es gilt $K=2$ und du willst die Gewichte berechnen, die zu $\alpha_k$ gehören. Du brauchst dafür $\alpha_{k-1}$ (für $P$ und $I_1$) und $\alpha_{k-2}$ (für $I_2$). Siehe Diagramm 4 im Bild der obigen Zelle. Was passiert im nächsten Schritt $k+1$? Dort brauchst du $\alpha_k$ und $\alpha_{k-1}$ um die Gewichte $P, I_1$ und $I_2$ für $\alpha_{k+1}$ zu berechnen. Die Information über $\alpha_{k-2}$ wird nie wieder benötigt, da die Reichweite $K=2$ ist!Wir müssen also an jedem Schnitt zwischen zwei Zeitschritten immer exakt die letzten $K=2$ Zustände in unserem „Arbeitsspeicher“ mitführen. Dieses Informationspaket ist das Bond $c$.

- Das eingehende Fenster (bevor wir $\alpha_k$ verarbeiten) ist: $c_{k-1} = (\alpha_{k-2}, \alpha_{k-1})$
- Das ausgehende Fenster (nachdem wir $\alpha_k$ verarbeitet haben) ist: $c_k = (\alpha_{k-1}, \alpha_k)$

Ein Bond $c_m$ ist also nichts anderes als ein Sammel-Index. Wenn $\alpha$ jeweils 16 mögliche Werte annehmen kann, dann gibt es $16 \times 16 = 256$ mögliche Kombinationen für das Fenster $c$. Das Bond hat die Dimension $16^K = 256$, da jeder der $K$ Einträge im Fenster 16 mögliche Werte annehmen kann. D.h $c_m$ gibt die Menge an Pfaden im Gedächnisfenster an.

Schritt 3: Konstruktion der Update-Maschine $G^{(k)}$: Wir lesen die vergangenen Zustände $(\alpha_{k-K}, \dots, \alpha_{k-1})$ nun direkt aus dem eingehenden Bond $c_{k-1}$ ab. Das physikalische Gewicht $W$ hängt damit effektiv nur noch vom neuen Zustand $\alpha_k$ und der komprimierten Vergangenheit $c_{k-1}$ ab. Dieses Gewicht und die Filter-Logik verschmelzen wir zu unserem dreibeinigen Tensor $G^{(k)}$. (Anmerkung: Für $k=1$ absorbiert der Tensor den Start-Halbschritt, und $\alpha_0$ übernimmt formal die Rolle von $c_0$).$$\begin{aligned} &= \sum_{\alpha_1\ldots\alpha_N} \sum_{c_1\dots c_N} (P_h)_{\beta\alpha_N} \left[ \prod_{k=1}^{N} \underbrace{W(\alpha_k, c_{k-1}) \cdot \mathbb{1}\big[\,c_k = (\alpha_{k-K+1}, \dots, \alpha_k)\,\big]}_{=\; G^{(k)}_{c_{k-1},\,\alpha_k,\,c_k}} \right] \\ &= \sum_{\alpha_1\ldots\alpha_N} \sum_{c_1\dots c_N} (P_h)_{\beta\alpha_N} \Big[ G^{(N)}_{c_{N-1},\alpha_N,c_N} \cdots G^{(2)}_{c_1,\alpha_2,c_2} \cdot G^{(1)}_{\alpha_0,\alpha_1,c_1} \Big] \end{aligned}$$

Der Tensor $G^{(k)}$ ist die mathematische Maschine, die den Schritt von $c_{k-1}$ zu $c_k$ durchführt. Er hat drei „Beine“ (Indizes):
- $c_{k-1}$ (Eingang): Das Wissen über die Vergangenheit (hier: $\alpha_{k-2}, \alpha_{k-1}$).
- $\alpha_k$ (Lokal): Der Zustand, den das System jetzt einnimmt.
- $c_k$ (Ausgang): Das aktualisierte Wissen, das in die Zukunft gereicht wird (hier: $\alpha_{k-1}, \alpha_k$).
- Die Dimension von $G^{(k)}_{\alpha_{k-1},\alpha_k,c_k}$ ist $16 \times 16 \times 16^K$, da $\alpha_0$ (Startzustand) und $\alpha_1$ (Lokaler Zustand bei $k=1$) von 0 bis 16 gehen, und $c_k$ alle möglichen Pfade im Gedächnisfenster $K$ enthält (d.h alle Kombis von $\alpha$ zwischen $\alpha_k$ und $\alpha_{k-K}$). Die Summe Reduziert den Ausdruck (für $\mathcal{L}(t_N)_{\beta\,\alpha_0}$) auf eine Matrix der größe $(\beta \alpha_0)$, sprich $16 \times 16$.

Ohne die Delta-Funktion $\mathbb{1}\big[\,c_k = (\alpha_{k-K+1}, \dots, \alpha_k)\,\big]_{=\; G^{(k)}_{c_{k-1},\,\alpha_k,\,c_k}}$ würde der algorithmus zur Berechnung eines Pfades die Historien aller anderen Pfade hernehmen. Die Historie eines Pfades soll aber nur von seiner eigenen Vorgeschichte abhängen. Wir machen das so weil man die Dynamik damit als Matrix-Vektor multiplikation schreiben kann, was sehr schnell zu berechnen geht.

Schritt 4: Faktorisierung durch das Distributivgesetz (Die Kontraktion): Jetzt greift der enorme Vorteil der Tensornetzwerke: Da $G^{(1)}$ nur von $\alpha_1$ abhängt, $G^{(2)}$ nur von $\alpha_2$ usw., können wir die gigantische äußere Summe $\sum_{\alpha_1 \dots \alpha_N}$ "in die Klammern hineinziehen". Wir summieren an jedem Knoten $k$ nur noch lokal über den einen physikalischen Zustand $\alpha_k$ (wir kontrahieren die physikalischen Beine). Das Ergebnis dieser lokalen Summen sind reine Matrizen $M^{(k)}$, die fortan nur noch über die Bond-Indizes $c$ miteinander verbunden sind.$$\begin{aligned} &= \sum_{c_1\dots c_N} \underbrace{\Big( \sum_{\alpha_1} G^{(1)}_{\alpha_0,\alpha_1,c_1} \Big)}_{\equiv\; M^{(1)}_{\alpha_0 c_1}} \cdot \underbrace{\Big( \sum_{\alpha_2} G^{(2)}_{c_1,\alpha_2,c_2} \Big)}_{\equiv\; M^{(2)}_{c_1 c_2}} \cdots \underbrace{\Big( \sum_{\alpha_N} (P_h)_{\beta\alpha_N}\, G^{(N)}_{c_{N-1},\alpha_N,c_N} \Big)}_{\equiv\; \tilde{M}^{(N)}_{c_{N-1} c_N, \beta}} \end{aligned}$$

# SVD — die Bond-Dimension $\chi$ und die trunkierte Matrix

In der Fensterform (2) hat jedes Bond die naive Dimension $\dim(c_m)=16^K$, da es alle möglichen Pfade im Fenster enthält. In Wahrheit ist das augmentierte Tensor-Netzwerk aber **schwach korreliert** (die Influenz-Kopplungen $\eta_{\Delta k}$ klingen ab), sodass an jedem Schnitt nur wenige Richtungen wirklich Gewicht tragen. Die **Singulärwertzerlegung (SVD)** findet genau diese Richtungen und wirft den Rest weg. Hier leiten wir das Kriterium her, das im Code (`_svd_split`, `_truncate_rank`, `_compress`) implementiert ist, und erklären jede Vereinfachung.

In deiner Formel $\mathcal{L}(t_N)_{\beta\,\alpha_0} = \sum_{c_1\dots c_N} \underbrace{\Big( \sum_{\alpha_1} G^{(1)}_{\alpha_0,\alpha_1,c_1} \Big)}_{\equiv\; M^{(1)}_{\alpha_0 c_1}} \cdot \underbrace{\Big( \sum_{\alpha_2} G^{(2)}_{c_1,\alpha_2,c_2} \Big)}_{\equiv\; M^{(2)}_{c_1 c_2}} \cdots \underbrace{\Big( \sum_{\alpha_N} (P_h)_{\beta\alpha_N}\, G^{(N)}_{c_{N-1},\alpha_N,c_N} \Big)}_{\equiv\; \tilde{M}^{(N)}_{c_{N-1} c_N, \beta}}$  hast du die physikalischen Zustände $\alpha_1 \dots \alpha_N$ bereits aufsummiert. Dadurch ist aus dem Netzwerk schon eine kleine Matrix für Start/Ende geworden.Um den Schnitt (die Bipartition) und die Matrix $\Psi_{a,b}$ zu sehen, müssen wir gedanklich einen Schritt zurückgehen, bevor diese $\sum_{\alpha}$ ausgeführt werden. Wir schauen uns das unsummierte Netzwerk an, bei dem alle physikalischen Beine $\alpha$ noch "offen" nach draußen hängen (d.h nicht über sie summiert wurde). 

Machen wir das an einem konkreten, kurzen Beispiel mit $N=4$ Zeitschritten. Wir schneiden das Netzwerk genau in der Mitte, also bei $m=2$. Die volle Information aller Pfade (das unsummierte Tensornetzwerk $\Psi$) sieht für $N=4$ so aus: Die inneren Summen laufen nur noch über die Gedächtnis-Bonds $c$ und $\Psi_{\alpha_0, \alpha_1, \alpha_2, \alpha_3, \alpha_4}$ hat Shape (16, 16, 16, 16, 16) :$$\Psi_{\alpha_0, \alpha_1, \alpha_2, \alpha_3, \alpha_4} = \sum_{c_1, c_2, c_3} G^{(1)}_{\alpha_0,\alpha_1,c_1} \cdot G^{(2)}_{c_1,\alpha_2,c_2} \cdot G^{(3)}_{c_2,\alpha_3,c_3} \cdot G^{(4)}_{c_3,\alpha_4}$$

Wir zerteilen die Kette gedanklich am Kabel $c_2$. Das spaltet die Summe in einen linken und einen rechten Teil auf.Der linke Block $L$ (Vergangenheit) besteht aus den Sites 1 und 2. Wir kontrahieren (summieren) alle inneren Verbindungen dieses Blocks – in diesem Fall nur $c_1$.
$$L_{\underbrace{(\alpha_0, \alpha_1, \alpha_2)}_{\text{Super-Index } a},\; c_2} = \sum_{c_1} G^{(1)}_{\alpha_0,\alpha_1,c_1} \cdot G^{(2)}_{c_1,\alpha_2,c_2}$$
Das Ergebnis ist ein Block mit shape $(16*16*16, 16^k)$, der auf der einen Seite von drei physikalischen Zuständen abhängt (unserem neuen Zeilen-Index $a$) und auf der anderen Seite genau einen Stecker hat: $c_2$.

Der rechte Block $R$ (Zukunft) besteht aus den Sites 3 und 4. Wir kontrahieren alle inneren Verbindungen – hier $c_3$.$$R_{c_2,\; \underbrace{(\alpha_3, \alpha_4)}_{\text{Super-Index } b}} = \sum_{c_3} G^{(3)}_{c_2,\alpha_3,c_3} \cdot G^{(4)}_{c_3,\alpha_4}$$
Dieser Block hat shape $(16^k, 16*16)$ und nimmt den Stecker $c_2$ entgegen und hängt von den restlichen physikalischen Zuständen ab (unserem neuen Spalten-Index $b$).

Wenn wir nun den linken und rechten Block wieder zusammenstecken, wird die komplizierte Tensor-Kette zu einer ganz normalen Matrixmultiplikation die einen shape von $(16*16*16, 16*16)$ hat:$$\Psi_{a,b} = \sum_{c_2} L_{a,\, c_2} \cdot R_{c_2,\, b}$$

- $a = (\alpha_0, \alpha_1, \alpha_2)$: Das ist die Zeile der Matrix. Sie repräsentiert eine ganz bestimmte physikalische Vorgeschichte.
- $b = (\alpha_3, \alpha_4)$: Das ist die Spalte. Sie repräsentiert einen ganz bestimmten physikalischen Zukunftsverlauf.
- $c_2$: Das ist der Laufindex der Matrixmultiplikation.

Die naive Dimension (Länge) des Kabels $c_2$ ist $16^K$. Das bedeutet, in dieser abstrakten Formulierung wird die Verbindung zwischen "Vergangenheit $a$" und "Zukunft $b$" durch $16^K$ verschiedene Informationskanäle vermittelt. Bis zu diesem Punkt enthält die Formulierung noch überhaupt keine Näherung. Sie ist mathematisch exakt!

Die Näherung passiert erst im nächsten, aktiven Schritt, wenn wir die Singulärwertzerlegung (SVD) auf diese exakte Matrix $\Psi_{a,b}$ anwenden:  

$$\Psi = U \Sigma V^\dagger$$

Die SVD ordnet die Informationskanäle im Kabel $c_2$ nach ihrer Wichtigkeit. Die Diagonaleinträge der Matrix $\Sigma$ sind die Schmidt-Koeffizienten $\sigma_k$.Wenn du dir diese Werte anschaust, stellst du typischerweise fest:
- $\sigma_1 = 0.9$ (extrem wichtig)
- $\sigma_2 = 0.05$ (wichtig)
- ...
- $\sigma_{100} = 0.0000001$ (fast unbedeutend)
- $\sigma_{256} = 0$ (völlig leer)

Anstatt also das naive Bond $c_2$ mit seiner riesigen Dimension $16^K$ (z.B. 256) mitzuschleppen, kappen wir die SVD einfach nach den größten Werten ab. Die Anzahl der Werte, die wir behalten (weil sie deutlich größer als Null sind), ist unsere effektive Bond-Dimension $\chi$.

In Matrixelementschreibweise sieht das ganze wie folgt aus:

$$\Psi_{a,b} = \sum_{k=1}^{r} \sigma_k \, U_{a, k} \, V^\dagger_{k, b}$$Erst jetzt entscheiden wir uns im Code (_truncate_rank, _compress), nicht mehr bis zum vollen Rang $r$ (der naiven Dimension $16^K$) zu summieren. Da die Singulärwerte $\sigma_k$ aufgrund der abklingenden Bad-Kopplungen extrem schnell gegen Null fallen, kappen wir die Summe künstlich bei einem viel kleineren Index $\chi \ll 16^K$ ab:  $$\Psi_{a,b} \approx \sum_{k=1}^{\chi} \sigma_k \, U_{a, k} \, V^\dagger_{k, b}$$

Um abzuschätzen wie groß der Fehler ist, den man macht wenn man die summe trunkiert, kann man folgendes verwenden: Der Trunkierungsfehler ist **exakt** die Frobenius-Summe der weggeworfenen Singulärwerte,

$$\lVert\Psi-\Psi_\chi\rVert_F^2=\sum_{k>\chi}\sigma_k^2.$$

wobei $\Psi_\chi := \sum_{k=1}^{\chi}\sigma_k\,u_k\,v_k^\dagger.$ Man kann beweisen, dass diese Form der trunkierung optimal ist.

# Anwendung im code
In unserer Erklärung haben wir so getan, als würden wir erst die riesigen Blöcke $L$ und $R$ mit ihren gigantischen inneren Summen (über $c_1, c_3$ etc.) exakt ausrechnen und danach bei $c_2$ in der Mitte schneiden. Das macht man im Code niemals so, denn dann hätte man den Exponenten $16^K$ ja doch im Arbeitsspeicher!In Wirklichkeit macht man das iterativ (Schritt für Schritt):Man fängt ganz links an. Man multipliziert $G^{(1)}$ und $G^{(2)}$ zusammen. Das Bond $c_1$ wächst kurzzeitig an, aber man macht sofort eine SVD und schneidet es auf $\chi$ ab. Dann nimmt man dieses komprimierte Ergebnis, multipliziert $G^{(3)}$ dazu, das Bond $c_2$ wächst kurz, man macht wieder sofort eine SVD und schneidet ab.Man schiebt das Fenster also durch die Kette und wendet die SVD an jedem einzelnen Bond an, bevor die Matrizen zu groß werden. So bleibt die Dimension an jeder Stelle immer maximal $\chi$.

### A. Das Problem: Lokaler vs. Globaler Fehler
Das Eckart-Young-Theorem besagt: Wenn du eine Matrix hast und ihre kleinsten Singulärwerte abschneidest, ist das die mathematisch perfekteste Näherung, die du machen kannst.Das Problem: Du hast nicht nur eine Matrix, sondern ein ganzes Netzwerk aus Matrizen ($G^{(1)} \cdot G^{(2)} \dots$). Wenn du jetzt mitten im Netzwerk bei Tensor $G^{(m)}$ lokal ein paar Singulärwerte wegschneidest, machst du lokal den kleinstmöglichen Fehler. Aber was, wenn die Tensoren rechts und links davon wie gigantische Lupen (Multiplikatoren) wirken? Dann könnte dein winziger lokaler Fehler global explodieren und die ganze Wellenfunktion zerstören!

### B. Die Gauge-Freiheit (Der $X X^{-1}$ Trick)
Bei Matrix-Multiplikationen darfst du jederzeit eine Einheitsmatrix $\mathbb{1}$ einschieben. Da $\mathbb{1} = X \cdot X^{-1}$ ist, kannst du das schreiben als:$$A \cdot B = (A \cdot X) \cdot (X^{-1} \cdot B)$$Das bedeutet: Du kannst Zahlenwerte, Gewichte und Konstanten völlig frei von Tensor $A$ in Tensor $B$ verschieben, ohne das globale physikalische Endresultat zu verändern. Diese Freiheit nennt man Gauge-Freiheit (Eichfreiheit).

### C. Isometrie (Das "Einfrieren" der Umgebung)
Wir nutzen diesen $X X^{-1}$ Trick nun systematisch aus. Wir wählen die Matrix $X$ nicht zufällig, sondern so, dass die Tensoren eine spezielle Eigenschaft bekommen: Isometrie.Eine isometrische Matrix erhält Längen und Abstände (wie eine reine Drehung im Raum). Wenn du einen Vektor mit einer isometrischen Matrix multiplizierst, ändert sich seine Norm (seine „Größe“) nicht.Wenn wir das Netzwerk durch geschickte Wahl von $X$ so umbauen, dass:
- alles links von unserem Schnitt links-isometrisch ist,
- alles rechts von unserem Schnitt rechts-isometrisch ist,

dann passiert die Magie: Die Umgebung wirkt nicht mehr wie eine Lupe! Da Isometrien die Norm erhalten, ist der Fehler, den du lokal an deinem Bond $c_m$ beim Abschneiden machst, exakt identisch mit dem Fehler der gesamten globalen Wellenfunktion $\Psi$.Die lokalen $\sigma_k$ spiegeln jetzt das echte, unbeeinflusste globale Gewicht des Pfades wider.

### D. Der Sweep-Algorithmus (_compress)
Genau das macht die Funktion _compress im Code. Sie geht nicht einfach blind hin und schneidet ab, sondern macht zwei Durchläufe (Sweeps):
- Der Präparations-Lauf (Links $\to$ Rechts mit QR-Zerlegung): Der Algorithmus geht von links nach rechts durch die Kette. Er nutzt die QR-Zerlegung (ein Standard-Verfahren der linearen Algebra), um systematisch den $X X^{-1}$ Trick anzuwenden. Er zwingt jeden Tensor in die links-isometrische Form und schiebt allen „numerischen Müll“ und alle Vorfaktoren immer weiter nach rechts. Am Ende dieses Durchlaufs ist das gesamte linke Netzwerk sauber normiert.
- Der Kompressions-Lauf (Rechts $\to$ Links mit SVD): Jetzt geht der Algorithmus rückwärts. Da die linke Seite jetzt eine perfekte Isometrie ist, kann er an jedem Bond die SVD anwenden und die kleinsten $\sigma_k$ abschneiden. Er weiß jetzt zu 100 %, dass das lokale Wegwerfen von $\sigma_k$ global den absolut geringstmöglichen Fehler verursacht.


### Finden von $X$ mit dem QR-Algorithmus

- Schritt 1: Den Tensor zur Matrix formen (Reshaping): Der Tensor $G^{(k)}$ hat drei Beine: $c_{k-1}$ (links), $\alpha_k$ (unten/lokal) und $c_k$ (rechts).Um Standard-lineare-Algebra zu nutzen, formen wir ihn in eine 2D-Matrix um. Wir fassen alle "linken" Informationen zu einem Zeilen-Index zusammen und behalten das "rechte" Bond als Spalten-Index:$$M_{(c_{k-1}, \alpha_k),\, c_k} = G^{(k)}_{c_{k-1},\, \alpha_k,\, c_k}$$

- Schritt 2: Die QR-Zerlegung anwenden: Wir werfen diese Matrix $M$ nun in einen Standard-QR-Algorithmus (z. B. numpy.linalg.qr). Dieser Algorithmus zerlegt jede beliebige Matrix zwingend in das Produkt von zwei neuen Matrizen:$$M = Q \cdot R$$Die Mathematik garantiert uns dabei zwei wunderbare Eigenschaften:$Q$ ist isometrisch (orthogonal): Es gilt $Q^\dagger Q = \mathbb{1}$.$R$ ist eine obere Dreiecksmatrix: Sie sammelt alle Skalierungen, Normen und (wie zuvor erwähnt) den "numerischen Müll" auf.

- Schritt 3: Wo ist unser $X$?: Erinnern wir uns an deinen $X X^{-1}$ Trick:Wir wollen den alten Tensor $G^{(k)}$ (unsere Matrix $M$) durch Multiplikation mit $X$ so verändern, dass er links-isometrisch wird.Wir wissen, dass $Q$ bereits die perfekte links-isometrische Matrix ist, die wir haben wollen! Wir definieren also unseren neuen, bereinigten Tensor einfach als $Q$:$$\tilde{G}^{(k)} = Q$$Wenn wir die QR-Gleichung $M = Q \cdot R$ nach $Q$ umstellen, erhalten wir:$$Q = M \cdot R^{-1}$$Vergleichen wir das nun mit deiner Vorschrift $G^{(k)} \to G^{(k)}X$:$$\tilde{G}^{(k)} = M \cdot \underbrace{R^{-1}}_{=\;X}$$Da haben wir es! Die gesuchte Matrix $X$ ist schlicht und ergreifend die Inverse der $R$-Matrix ($X = R^{-1}$).

- Schritt 4: Den Müll nach rechts schieben: Der zweite Teil deines Tricks besagt, dass der nachfolgende Tensor $G^{(k+1)}$ zum Ausgleich mit $X^{-1}$ multipliziert werden muss:$$\tilde{G}^{(k+1)} = X^{-1} \cdot G^{(k+1)}$$Da $X = R^{-1}$ ist, gilt logischerweise $X^{-1} = R$.Das bedeutet, wir müssen nicht einmal invertieren! Wir nehmen einfach direkt die $R$-Matrix, die uns der QR-Algorithmus ausgespuckt hat, und multiplizieren sie in das linke Bein des nächsten Tensors hinein:$$\tilde{G}^{(k+1)}_{c_k,\, \alpha_{k+1},\, c_{k+1}} = \sum_{c'_k} R_{c_k,\, c'_k} \cdot G^{(k+1)}_{c'_k,\, \alpha_{k+1},\, c_{k+1}}$$

### Der "Bootstrap"-Prozess (Das Bauen der Kette) und das handlen von $\chi$
Stell dir vor, wir fangen bei der Zeit $k=1$ an und bauen die Kette von links nach rechts auf. Die lokale Dimension eines jeden physikalischen Zustands $\alpha_k$ sei wieder 16. Wir haben eine festgelegte Toleranzgrenze $\epsilon$ (z. B. $10^{-8}$), bei der wir abschneiden wollen.

Schritt 1: Der Start ($k=1$)Wir nehmen den allerersten Tensor $G^{(1)}$. Dieser hat ein offenes Bein für den Startzustand $\alpha_0$ und eines für $\alpha_1$. Die Dimension dieses ersten Bonds $c_1$ ist exakt $16$.
- Dimension im Speicher: 16
- Das ist winzig. Wir müssen hier noch gar nichts abschneiden. Das Bond $c_1$ wandert einfach mit der Dimension 16 in den nächsten Schritt.

Schritt 2: Das erste Wachstum ($k=2$)Jetzt multiplizieren wir den nächsten physikalischen Schritt $G^{(2)}$ dazu. Dieses neue Element hat selbst 16 Zustände ($\alpha_2$). Der Zustandsraum für das neue Bond $c_2$ ergibt sich aus dem vorherigen Bond (16) multipliziert mit dem neuen Zustand (16).
- Dimension vor SVD: $16 \times 16 = 256$.
- Eine Matrix der Größe 256 ist für den Computer ein Witz. Wir führen nun die SVD auf dieser Matrix aus.
- Die Trunkierung: Die SVD liefert 256 Singulärwerte. Wir schauen, wie viele davon über unserer Grenze $\epsilon$ liegen. Nehmen wir an, das sind 50.
- Ergebnis: Unser neues Bond $c_2$ hat jetzt die Dimension $\chi = 50$. Wir haben das $256$-dimensionale Bond auf 50 komprimiert.

Schritt 3: Der entscheidende Moment ($k=3$)Jetzt fügen wir $G^{(3)}$ hinzu (wieder 16 lokale Zustände).Und hier passiert die Magie: Wir rechnen jetzt nicht $16 \times 16 \times 16 = 4096$.Wir nehmen unser bereits gestutztes Bond $c_2$ (das ja nur noch Dimension 50 hat) und multiplizieren es mit den neuen 16 Zuständen.
- Dimension vor SVD: $50 \times 16 = 800$.
- Auch eine Matrix der Größe 800 ist absolut problemlos berechenbar. Wir machen wieder sofort eine SVD auf diesen 800 Elementen.
- Die Trunkierung: Die SVD liefert 800 Singulärwerte. Wir werfen alles unter $\epsilon$ weg. Sagen wir, es bleiben 85 relevante Werte übrig.
- Ergebnis: Unser neues Bond $c_3$ hat nun die Dimension $\chi = 85$.

Du siehst das Muster: Die Dimension des Bonds vor dem Abschneiden ist niemals $16^K$. Sie ist immer nur:$$\text{Dimension vor SVD} = \chi_{\text{alt}} \times 16$$Wir lassen den Zustandsraum in jedem Schritt immer nur um den Faktor 16 anwachsen und zwingen ihn durch die SVD sofort wieder in eine komprimierte Form zurück, bevor wir den nächsten Schritt hinzufügen. 

D.h man fixiert **nicht** $\chi$, sondern eine relative Genauigkeit $\varepsilon$ und lässt $\chi$ so klein wie möglich werden. Mit Eckart–Young ist der relative Fehler

$$\frac{\lVert\Psi-\Psi_\chi\rVert_F}{\lVert\Psi\rVert_F}=\sqrt{\frac{\sum_{k>\chi}\sigma_k^2}{\sum_k\sigma_k^2}}\ \le\ \varepsilon .$$

Aufgelöst nach dem kleinstmöglichen $\chi$ ergibt sich die **exakt implementierte Formel** (`_truncate_rank`):

$$\boxed{\;\chi=\min\Big\{\,r\in\{1,\dots,\chi_{max}\}\ :\ \sum_{k>\chi}\sigma_k^2\ \le\ \varepsilon^2\sum_{k}\sigma_k^2\,\Big\},\qquad \chi\leftarrow\min(\chi,\chi_{\max}).\;}$$

Konkret zählt der Code, wie viele der **kleinsten** Singulärwerte man verwerfen darf, bis die kumulierte verworfene Frobenius-Masse die Schwelle $\varepsilon^2\lVert M\rVert_F^2$ erreicht (`np.cumsum(s2[::-1])`, `np.searchsorted`).

### 5. Was noch vereinfacht wird, damit man es rechnen kann

Das Problem ist simpel: Eine Standard-SVD berechnet alle Singulärwerte und -vektoren einer Matrix $M$. Wenn $M$ eine $4000 \times 4000$ Matrix ist, rechnet der Algorithmus $\mathcal{O}(D^3)$ Operationen, um 4000 exakte Singulärwerte zu finden – nur damit du am Ende sagst: „Danke, ich brauche aber nur die 50 größten für mein $\chi$, den Rest werfe ich weg.“ Das ist extreme Ressourcenverschwendung.Die rSVD (nach Halko, Martinsson, Tropp) dreht den Spieß um: Wir suchen gezielt nur nach dem kleinen, dominanten Unterraum, in dem die meiste physikalische Information (die größte Varianz) liegt.Hier ist die detaillierte Aufschlüsselung, was die vier Schritte und die zwei Korrekturen wirklich machen:

- 1. Skizze (Das randomisierte Abtasten): Anstatt die Matrix $M$ exakt zu analysieren, tasten wir sie mit Zufallsvektoren ab. Wir erstellen eine schmale Matrix $\Omega$, die aus reinem Gaußschen Rauschen besteht.Ihre Breite $\ell$ ist unsere Schätzung für die Bond-Dimension $\chi_{\text{hint}}$, plus ein Sicherheitsaufschlag („Oversampling“, hier $1.3\times$ plus 64).Wenn wir $M \cdot \Omega$ rechnen, projizieren wir die gigantische Matrix $M$ auf diese zufälligen Vektoren. Da Gauß-Vektoren in alle Richtungen des Raumes zeigen, fangen sie die Dimensionen, in denen $M$ „groß“ ist (die dominanten Zustände), automatisch am stärksten ein.

- 2. Power-Iteration (Die Lupe für physikalische Zustände): Das ist der intelligenteste Schritt. Oft hat ein Quantensystem ein Gedächtnis, dessen Singulärwerte $\sigma$ nur ganz flach abfallen (z. B. $\sigma_1 = 1.0, \sigma_2 = 0.95, \sigma_3 = 0.9$). Das Rauschen der Gauß-Matrix $\Omega$ könnte diese Werte vermischen.Multiplizieren wir die Matrix jedoch mit sich selbst, also $M \cdot M^\dagger \cdot M$, potenzieren sich die Singulärwerte auf $\sigma_i^3$.$1.0^3 = 1.0$$0.9^3 = 0.729$Der physikalische Abstand zwischen den dominanten Zuständen und dem „Rest“ wird künstlich und massiv vergrößert. Die Zufallsvektoren in $Y$ werden durch diese Power-Iteration wie von einem Magneten extrem stark in den dominanten Unterraum gezogen.

- 3. Range-Finder (Das Destillieren der Basis): Die Vektoren in unserer Matrix $Y$ spannen jetzt mit extrem hoher Wahrscheinlichkeit genau den wichtigen physikalischen Unterraum von $M$ auf. Das Problem: Sie sind nicht orthonormal (also stehen nicht im 90-Grad-Winkel zueinander) und teilweise redundant.Die QR-Zerlegung $Q = \mathrm{qr}(Y)$ bereinigt das. Sie gibt uns eine saubere, orthonormale Basis $Q$ für genau diesen Unterraum. Die riesige Matrix $M$ lässt sich nun durch $Q \cdot Q^\dagger \cdot M$ fast perfekt annähern.

- 4. Projektion (Das eigentliche Rechnen): Jetzt kommt der Performance-Gewinn. Wir projizieren die riesige Matrix $M$ in unsere gefundene, winzige Basis $Q$:$$B = Q^\dagger M$$Die Matrix $B$ ist winzig (nämlich nur $\ell \times n$). Auf diese kleine Matrix $B$ werfen wir nun den normalen, exakten SVD-Algorithmus aus der linearen Algebra ($B = U_B \Sigma V^\dagger$). Da $B$ so klein ist, kostet das fast keine Rechenzeit.Die wichtigen Schmidt-Werte $\Sigma$ und die Zukunfts-Zustände $V^\dagger$ haben wir damit sofort. Die Vergangenheits-Zustände $U$ im großen Raum bekommen wir durch simples Zurücktransformieren: $U = Q U_B$.

Die Ehrlichkeits-Korrekturen (Fehlerkontrolle): Da wir die kleinen Singulärwerte gar nicht erst berechnet haben, könnten wir ein Problem mit unserem Abbruchkriterium $\varepsilon$ bekommen. Der Code repariert das durch zwei geniale Sicherheitsnetze:
- Korrektur 1: Der Tail-Term (Nichts geht verloren)Das Eckart-Young-Kriterium verlangt, dass wir die Summe aller weggeworfenen $\sigma^2$ kennen. In der rSVD kennen wir diese aber nicht, da wir sie nie berechnet haben!Wir nutzen einen Trick aus der linearen Algebra: Die quadrierte Frobenius-Norm einer Matrix $\lVert M \rVert_F^2$ (die einfache Summe aller quadrierten Matrixeinträge) ist exakt gleich der Summe aller quadrierten Singulärwerte.Die Berechnung von $\lVert M \rVert_F^2$ ist algorithmisch spottbillig. Der Code rechnet also:$$\text{Weggeworfene Masse (Tail)} = \underbrace{\lVert M \rVert_F^2}_{\text{Exakte Gesamtmasse}} - \underbrace{\sum_{k=1}^{\ell} \sigma_k^2}_{\text{Gefundene Masse aus rSVD}}$$Dieser „Tail“ wird dann einfach auf unseren berechneten Trunkierungsfehler draufaddiert. So tricksen wir uns nicht selbst aus und das $\varepsilon$-Kriterium bleibt mathematisch strikt und ehrlich.
- Korrektur 2: Adaptivität: Was passiert, wenn unsere anfängliche Schätzung für $\chi$ viel zu klein war, weil das Quantensystem plötzlich massive Verschränkung (Korrelation) aufbaut?Der Code merkt das: Wenn fast alle Singulärwerte, die wir im Unterraum der Größe $\ell$ gefunden haben, wichtig sind ($\chi \ge \ell - 8$), dann war unser Netz zu klein ausgeworfen. In diesem Fall verdoppelt der Algorithmus einfach die Größe der Skizze $\Omega$ und probiert es nochmal, bevor er falsche Physik simuliert.

### 6. Ergebnis: $\chi$ und die Endkosten

Nach der Trunkierung trägt jedes Bond nur das **effektive** $\chi$ — für dieses FMO-Problem sättigt es bei $\chi\approx100\text{–}210$, gegenüber $16^{K}=16^{25}\approx10^{30}$ naiv. Die Gesamtkosten werden damit **polynomial**,

$$\mathcal O\big(N\,K\,\chi^3\big)\quad\text{statt}\quad 16^N,$$

und die exponentiell große Pfadsumme (1) wird tatsächlich rechenbar. $\varepsilon$ (Genauigkeit) und $\chi_{\max}$ (harte Schranke) sind die beiden Stellschrauben; kleineres $\varepsilon$ ⇒ größeres $\chi$ ⇒ genauer, aber teurer.

| Objekt hier | im Code (`pathintegral_map.py`) |
|---|---|
| Schmidt-Werte $\sigma_k$, Rang $\chi$ | `s`, `keep` in `_truncate_rank` |
| Kriterium $\sum_{k>\chi}\sigma_k^2\le\varepsilon^2\lVert M\rVert_F^2$ | `_truncate_rank(s, eps, chi_max, total2)` |
| kanonische Form (QR-Sweep + SVD-Sweep) | `_compress` |
| randomisierte SVD + Power-Iteration + Tail | `_svd_split` (der `while`-Zweig) |
| harte Schranke $\chi_{\max}$, Toleranz $\varepsilon$ | `chi_max`, `eps` in `PathIntegralMap.__init__` |

In [ ]:
def _step(self, mps):
    """
    Die Herzstueck-Funktion ("Zip-Up" Algorithmus).
    Fuegt den neuen Zeitschritt alpha_k hinzu und baut die neuen
    Tensoren auf. Dies geschieht nach dem "Ziehharmonika"-Prinzip:
    Wir multiplizieren die Matrizen und trunkieren sofort per SVD auf chi,
    noch BEVOR die naechste Operation den Zustandsraum sprengen kann.
    So existiert das 16^K Bond niemals im Arbeitsspeicher.
    """
    D = self.D
    n_old = len(mps)                                # Die aktuelle Länge unserer MPS-Kette (wie viele Zeitschritte wir uns gerade merken).
    W = self.Imats[1] * self.P * self.I0[:, None]   # W(alpha_k, c_k-1): Gewicht für den kürzesten Pfad-Schritt: W[b, a] = I_1[b,a] P[b,a] I0[b]

    if n_old == 1:
        # Spezialfall: Der allererste Schritt. Wenn das Netzwerk erst einen Tensor lang ist, gibt es noch kein Gedächtnis zum Durchlaufen.
        G = mps[0][:, :, 0]                   # (w, a)
        Y = G[:, :, None] * W.T[None, :, :]   # (w, a, b), Der Startzustand G wird mit dem lokalen Gewicht W multipliziert
        w, a_, b_ = Y.shape
        U, s, Vh = _svd_split(Y.reshape(w * a_, b_), self.eps, self.chi_max)  # Das Ergebnis Y wird direkt per SVD in zwei Tensoren zerlegt.
        mps = [U.reshape(w, a_, -1), (s[:, None] * Vh).reshape(-1, b_, 1)]
    else:
        # Zip-up von Links nach Rechts durch das Gedaechtnis-Fenster, left end (oldest slice, distance n_old)
        G = mps[0]                            # (w, a, x), G ist der älteste Tensor in unserem Gedächtnis.
        F = self.Imats[n_old]                 # F ist der Bad-Einfluss für die maximal mögliche Distanz n_old. Das ist die Wechselwirkung zwischen dem ältesten und dem brandneuen (noch zu erstellenden) Zeitschritt.
        Y = G[:, :, None, :] * F.T[None, :, :, None]     # (w, a, l, x), Wir multiplizieren G mit F. Hierbei entsteht ein neuer Index l in der Mitte von Y (Dimension 2). Das ist der physikalische Index des neuen Zeitschritts
        w, a_, l_, x_ = Y.shape
        U, s, Vh = _svd_split(Y.reshape(w * a_, l_ * x_),self.eps, self.chi_max, rank_hint=x_)  # Wir formen Y in eine Matrix um und zerlegen sie per SVD.
        new_mps = [U.reshape(w, a_, -1)]            # Der linke Teil U wird unser aktualisierter, ältester Tensor und kommt in die neue Liste new_mps.
        C = (s[:, None] * Vh).reshape(-1, l_, x_)   # (c, l, x), Der rechte Teil (Singulärwerte $s$ mal Basis $V^\dagger$) wird zu unserem "Gepäck" C (Carry). Wichtig: Dieses Gepäck C trägt den neuen Index l_ in sich! Wir schieben ihn nun durch das gesamte Netzwerk.

        # middle sites (distances n_old-1 .. 2)
        # Hier greift das Verschieben des "l" Index durch das gesamte Netz
        for pos in range(1, n_old - 1):             # Wir iterieren durch die inneren Tensoren.
            F = self.Imats[n_old - pos]             # F ist der Bad-Einfluss für die exakte zeitliche Distanz dieses Tensors zur Gegenwart.
            G = mps[pos]                            # (x, a, y), 
            Y = np.tensordot(C, G, axes=([2], [0])) # (c, l, a, y), Wir saugen das Gepäck C (das von links kommt und den Index l enthält) in den aktuellen Tensor G auf.
            Y *= F[None, :, :, None]                # F[l, a], Wir applizieren den Bad-Einfluss F. F verknüpft den lokalen physikalischen Zustand a mit unserem durchgeschleiften Gegenwarts-Zustand l.
            c_, l_, a_, y_ = Y.shape
            M = Y.transpose(0, 2, 1, 3).reshape(c_ * a_, l_ * y_)       # Wir transponieren Y. Warum? Wir wollen den linken Bond c_ und den lokalen physikalischen Index a_ auf der linken Seite des Schnitts behalten. Der durchgeschleifte Gegenwarts-Index l_ und der rechte Bond y_ müssen auf die rechte Seite des Schnitts, damit sie mit dem Gepäck C weiterreisen!
            U, s, Vh = _svd_split(M, self.eps, self.chi_max, rank_hint=y_)  # Sofortiges Beschneiden (Truncation) zurueck auf effektives chi
            new_mps.append(U.reshape(c_, a_, -1))                       
            C = (s[:, None] * Vh).reshape(-1, l_, y_)           # Nach der SVD speichern wir U als neuen mittleren Tensor ab. C wandert weiter nach rechts.

        # last old site (distance 1) + new slice b  (l == b)
        # Abschluss am rechten Rand, wo die Indikator-Logik des Schieberegisters greift
        G = mps[-1][:, :, 0]                             # (x, a),    Wir holen den bisher "neuesten" Tensor G aus der alten Kette (wir schneiden den leeren rechten Bond weg).
        Y = np.tensordot(C, G, axes=([2], [0]))          # (c, b, a)  Wir kontrahieren unser Gepäck C (das immer noch den Zukunfts-Index b bzw. l trägt) hinein.
        Y *= W[None, :, :]                               # W[b, a]    Wir multiplizieren mit W – dem Kurzzeit-Propagator und Nächste-Nachbarn-Einfluss (den wir in Zeile 1 definiert haben).
        c_, b_, a_ = Y.shape
        M = Y.transpose(0, 2, 1).reshape(c_ * a_, b_)        # Wir sortieren die Indizes: Alter physikalischer Index a_ nach links, brandneuer Gegenwarts-Index b_ nach rechts.
        U, s, Vh = _svd_split(M, self.eps, self.chi_max)     # Die SVD führt den allerletzten Schnitt aus.
        new_mps.append(U.reshape(c_, a_, -1))                # U wird der vorletzte Tensor.
        new_mps.append((s[:, None] * Vh).reshape(-1, b_, 1)) # Das restliche Gepäck ($s \cdot V^\dagger$) bildet den brandneuen Tensor ganz rechts außen. Das Netzwerk ist nun offiziell um einen Zeitschritt gewachsen!
        mps = new_mps

    # restore right-canonical gauge (centre back to site 0), truncating
    # at proper cuts (left part is left-canonical from the pass above)
    # SVD Sweep rueckwaerts, um Fehler gloabal minimal zu halten und
    # das System fuer den naechsten Schritt zu praeparieren.
    for i in range(len(mps) - 1, 0, -1):   # Am Ende ruht das Orthogonalitätszentrum (die Norm des Quantensystems) wieder perfekt auf dem ersten Tensor mps[0].
        l, p, r = mps[i].shape  
        U, s, Vh = _svd_split(mps[i].reshape(l, p * r), self.eps, self.chi_max)
        mps[i] = Vh.reshape(-1, p, r)
        mps[i - 1] = np.tensordot(mps[i - 1], U * s[None, :], axes=([2], [0]))      # Am Ende ruht das Orthogonalitätszentrum (die Norm des Quantensystems) wieder perfekt auf dem ersten Tensor mps[0].

    # memory truncation: marginalize slices older than kmax
    # Hier implementieren wir den "Cutoff". Das Gedaechtnis reicht nur
    # kmax Schritte zurueck. Der aelteste Zustand (alpha_{k-K}) wird
    # ueber die Funktion .sum() physikalisch "vergessen" (auskontrahiert).
    if len(mps) > self.kmax:            # Wenn unsere Kette länger geworden ist als die definierte Gedächtnistiefe kmax des Bades.
        v = mps[0].sum(axis=1)          # (alpha_0 bond, y), Wir berechnen die mathematische Teilspur (Trace) über die physikalischen Freiheitsgrade des ältesten Tensors. Physikalisch bedeutet das: Wir ignorieren ab sofort die exakten Quantenzustände zu diesem Zeitpunkt, weil ihr Einfluss auf die Gegenwart komplett abgeklungen ist.
        mps[1] = np.tensordot(v, mps[1], axes=([1], [0]))   # Der verbleibende Vektor v (der Rand des Netzwerks) wird einfach auf den Bond des nächsten Tensors mps[1] aufgeschlagen.
        mps.pop(0)                      # Der älteste Tensor wird gelöscht. Das Netzwerk behält eine konstante Länge und der Speicher läuft nicht voll.
    return mps

# Das hier bringt eigentlich nicht so viel. Siehe chat

## Route C — the clock register: every time from one run, one measurement

Implementation of §8. `flag_oracle` is $F_j$, `controlled_channel` turns a step into its flag-controlled
version, and `clock_trajectory` assembles the circuit. Nothing but the final `save_density_matrix` reads
anything, and that single read-out returns the joint clock–system state whose diagonal is
$\frac{1}{n+1}\rho(t)$ for every $t$ at once.

In [ ]:
def flag_oracle(n_clock, j):
    """F_j : |t>|f> -> |t>|f XOR [t>=j]>.  Index = f*2^n_clock + t, i.e. the flag
    is the high bit, so append it to (clock qubits, flag)."""
    n = 1 << n_clock
    F = np.zeros((2 * n, 2 * n))
    for t in range(n):
        if t >= j:
            F[t, n + t] = F[n + t, t] = 1.0        # swap f=0 <-> f=1
        else:
            F[t, t] = F[n + t, n + t] = 1.0
    return F


def controlled_channel(U):
    """|f> (x) |x>  ->  |f> (x) U^f |x>.  Flag is again the high bit."""
    n = U.shape[0]
    C = np.eye(2 * n, dtype=complex)
    C[n:, n:] = U
    return C


def clock_trajectory(kraus, d, n_clock, rho0=None):
    """§8 for the deterministic route: ONE circuit, ONE measurement at the end,
    returning rho(t) for every t = 0 .. 2^n_clock - 1.

    Returns (pops, weights, circuit): `pops[t]` are the populations at time t and
    `weights[t]` = P(clock = t), which is flat by construction."""
    U_S, n_sys, n_env = stinespring_unitary(kraus, d)
    n_steps = (1 << n_clock) - 1
    if rho0 is None:
        rho0 = np.zeros((1 << n_sys, 1 << n_sys), complex); rho0[0, 0] = 1.0

    sysR = QuantumRegister(n_sys, 'sys');  envR = QuantumRegister(n_env, 'env')
    clkR = QuantumRegister(n_clock, 'clk'); flgR = QuantumRegister(1, 'flag')
    qc = QuantumCircuit(sysR, envR, clkR, flgR)

    zero = lambda k: np.diag([1.0] + [0.0] * ((1 << k) - 1)).astype(complex)
    qc.set_density_matrix(DensityMatrix(
        np.kron(zero(1), np.kron(zero(n_clock), np.kron(zero(n_env), rho0)))))
    for q in clkR:
        qc.h(q)                                    # uniform superposition over t

    cU = UnitaryGate(controlled_channel(U_S), label='c-U_S')
    for j in range(1, n_steps + 1):
        Fg = UnitaryGate(flag_oracle(n_clock, j), label=f'>={j}')
        qc.append(Fg, list(clkR) + [flgR[0]])      # compute the predicate
        qc.append(cU, list(sysR) + list(envR) + [flgR[0]])
        qc.append(Fg, list(clkR) + [flgR[0]])      # uncompute it
        qc.reset(envR)
    qc.save_density_matrix(list(sysR) + list(clkR), label='joint')

    rho = np.asarray(AerSimulator(method='density_matrix')
                     .run(qc).result().data()['joint'])
    P = np.real(np.diag(rho)).reshape(1 << n_clock, 1 << n_sys)   # index = t*2^n_sys + i
    w = P.sum(axis=1)
    return P[:, :d] / w[:, None], w, qc

In [ ]:
N_CLOCK = 5                       # 2^5 = 32 time points, i.e. t = 0 .. 310 fs
t0 = time.time()
pops_C, w_C, qc_C = clock_trajectory(res['Lindblad']['kraus_dt'], d, N_CLOCK)
n_C = (1 << N_CLOCK) - 1

ref_C = res['Lindblad']['pops'][:n_C + 1]
print(f'circuit : {qc_C.num_qubits} qubits, depth {qc_C.depth()}, '
      f'{n_C} steps -> {n_C + 1} time points, {time.time() - t0:.1f} s')
print(f'          ONE measurement at the end; the system is never read out in between')
print(f'clock   : P(t) flat to {abs(w_C - 1 / (1 << N_CLOCK)).max():.1e} '
      f'(uniform value {1 / (1 << N_CLOCK):.5f})')
print(f'accuracy: max |clock result - known trajectory| over all '
      f'{n_C + 1} times = {np.abs(pops_C - ref_C).max():.2e}')

### The same thing for the non-Markovian route

Route B under the clock, as derived at the end of §8: $U_H$ replaces $U_S$, each step gets its own
ancilla so the record survives to the end, and post-selecting all of them on $|0\rangle$ weights the
clock by $\prod_{j\le t}p_j$ without biasing the conditional populations.

In [ ]:
def clock_trajectory_memory(E, X0, n_clock, m, d):
    """§8 for the post-selected route. One circuit; the memory register is never
    measured, the clock and the n ancillas are read once at the end."""
    D = d * d
    n_steps = (1 << n_clock) - 1
    Hm, Qm = arnoldi(E, X0, m); m = Hm.shape[0]
    mp = 1 << int(np.ceil(np.log2(m)))
    A = np.zeros((mp, mp), complex); A[:m, :m] = Hm
    U_H, s = dilate(A)
    n_mem = int(np.log2(mp))

    memR = QuantumRegister(n_mem, 'mem'); ancR = QuantumRegister(n_steps, 'anc')
    clkR = QuantumRegister(n_clock, 'clk'); flgR = QuantumRegister(1, 'flag')
    qc = QuantumCircuit(memR, ancR, clkR, flgR)
    y0 = np.zeros(mp, complex); y0[0] = 1.0
    qc.initialize(y0, list(memR))
    for q in clkR:
        qc.h(q)

    cU = UnitaryGate(controlled_channel(U_H), label='c-U_H')
    for j in range(1, n_steps + 1):
        Fg = UnitaryGate(flag_oracle(n_clock, j), label=f'>={j}')
        qc.append(Fg, list(clkR) + [flgR[0]])
        qc.append(cU, list(memR) + [ancR[j - 1]] + [flgR[0]])   # its own ancilla
        qc.append(Fg, list(clkR) + [flgR[0]])
    qc.save_statevector(label='final')

    sv = np.asarray(AerSimulator(method='statevector').run(qc).result().data()['final'])
    sv = sv.reshape(2, 1 << n_clock, 1 << n_steps, mp)      # [flag, clk, anc, mem]
    branch = sv[0, :, 0, :]                                 # flag=0 AND every ancilla |0>
    weights = np.array([np.vdot(a, a).real for a in branch])

    # classical norm bookkeeping of §5, one scalar per time
    pops, yv, scale = [], y0.copy(), np.linalg.norm(X0)
    for t in range(1 << n_clock):
        pops.append(np.real(np.diag((Qm @ yv[:m] * scale)[:D].reshape(d, d, order='F'))))
        top = (U_H @ np.concatenate([yv, np.zeros(mp)]))[:mp]
        nn = np.linalg.norm(top); yv = top / nn; scale *= nn * s
    return np.array(pops), weights, qc

In [ ]:
NAME_C, N_CLOCK_B, M_B = 'Path integral', 3, 16
K_B = res[NAME_C]['K']
X0_B = np.concatenate([res[NAME_C]['rho'][i].flatten(order='F') for i in range(K_B)][::-1])

t0 = time.time()
pops_D, w_D, qc_D = clock_trajectory_memory(res[NAME_C]['E'], X0_B, N_CLOCK_B, M_B, d)
ref_D = res[NAME_C]['pops'][K_B - 1:K_B - 1 + len(pops_D)]

print(f'circuit : {qc_D.num_qubits} qubits, depth {qc_D.depth()}, '
      f'{len(pops_D)} time points, {time.time() - t0:.1f} s')
print(f'clock   : P(t) decays {w_D[0]:.5f} -> {w_D[-1]:.5f} '
      f'(accumulated post-selection), conditional populations unaffected')
print(f'accuracy: max |clock result - known trajectory| = '
      f'{np.abs(pops_D - ref_D).max():.2e}')

### The clock grid

Two steps of route C. `clk` carries $\sum_t|t\rangle$, `flag` holds the predicate $[t\ge j]$ computed by
$F_j$ (`>=1`, `>=2`, …) and uncomputed immediately after, and the box `c-U_S` is the channel step
controlled on that flag. The $|0\rangle$ boxes on `env` are the partial trace. There is no meter
anywhere in the circuit — the single measurement happens only at the very end, on `sys` and `clk`
together.

In [ ]:
U_S_show, ns_s, ne_s = stinespring_unitary(res['Lindblad']['kraus_dt'], d)
NC_SHOW, STEPS_SHOW = 3, 2
sysS = QuantumRegister(ns_s, 'sys'); envS = QuantumRegister(ne_s, 'env')
clkS = QuantumRegister(NC_SHOW, 'clk'); flgS = QuantumRegister(1, 'flag')
qc_show = QuantumCircuit(sysS, envS, clkS, flgS)
for q in clkS:
    qc_show.h(q)
cU_show = UnitaryGate(controlled_channel(U_S_show), label='c-U_S')
for j in range(1, STEPS_SHOW + 1):
    Fg = UnitaryGate(flag_oracle(NC_SHOW, j), label=f'>={j}')
    qc_show.append(Fg, list(clkS) + [flgS[0]])
    qc_show.append(cU_show, list(sysS) + list(envS) + [flgS[0]])
    qc_show.append(Fg, list(clkS) + [flgS[0]])
    qc_show.reset(envS)

fig, ax = plt.subplots(figsize=(13, 5))
qc_show.draw(output='mpl', ax=ax)
ax.set_title('Route C (§8) — the clock grid: $H^{\\otimes n_c}$ prepares $\\sum_t|t\\rangle$, '
             '$F_j$ computes $[t\\geq j]$ into the flag,\n'
             '`c-U_S` is the flag-controlled channel step, $|0\\rangle$ resets the environment. '
             f'Showing {STEPS_SHOW} of {(1 << N_CLOCK) - 1} steps; no meter until the very end.',
             fontsize=11, pad=16)
fig.tight_layout()
fig.savefig(f'pictures/clock_grid_N{N}.png', dpi=200); plt.show()
print(f'wrote pictures/clock_grid_N{N}.png')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
tC = np.arange(len(pops_C)) * (t_fs[1] - t_fs[0])
for i in range(d):
    axes[0].plot(tC, ref_C[:, i], color='k', ls='--', lw=1.2,
                 label='known trajectory' if i == 0 else None)
    axes[0].plot(tC, pops_C[:, i], 'o', ms=4, color=col['Lindblad'],
                 label='from the single clock measurement' if i == 0 else None)
axes[0].set_title(f'Route C, deterministic — all {len(pops_C)} times, one measurement\n'
                  f'{qc_C.num_qubits} qubits, deviation '
                  f'{np.abs(pops_C - ref_C).max():.1e}', fontsize=10)
axes[0].set_xlabel('time / fs'); axes[0].set_ylabel('population')
axes[0].grid(alpha=0.25); axes[0].legend(fontsize=8.5)

tD = (K_B - 1 + np.arange(len(pops_D))) * (t_fs[1] - t_fs[0])
for i in range(d):
    axes[1].plot(tD, ref_D[:, i], color='k', ls='--', lw=1.2,
                 label='known trajectory' if i == 0 else None)
    axes[1].plot(tD, pops_D[:, i], 'o', ms=5, color=col[NAME_C],
                 label='from the single clock measurement' if i == 0 else None)
ax2 = axes[1].twinx()
ax2.plot(tD, w_D, ':', color='#888888', lw=1.4)
ax2.set_ylabel('clock weight $P(t)$', color='#888888'); ax2.tick_params(colors='#888888')
axes[1].set_title(f'Route C, post-selected ({NAME_C}) — {len(pops_D)} times\n'
                  f'{qc_D.num_qubits} qubits, deviation '
                  f'{np.abs(pops_D - ref_D).max():.1e}', fontsize=10)
axes[1].set_xlabel('time / fs'); axes[1].set_ylabel('population')
axes[1].grid(alpha=0.25); axes[1].legend(fontsize=8.5, loc='center right')

fig.suptitle('Every state at every time, from one run ending in one measurement', fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(f'pictures/clock_trajectory_N{N}.png', dpi=200); plt.show()
print(f'wrote pictures/clock_trajectory_N{N}.png')

## 8. All times from one run and one final measurement: the clock register

Everything so far delivers $\rho(T)$ for one chosen $T$. We now show that the entire trajectory
$\{\rho(t)\}_{t=0}^{n}$ can be obtained from a **single execution ending in a single measurement**, at a
cost of only $\lceil\log_2(n+1)\rceil+1$ extra qubits. The construction is the Feynman–Kitaev clock
adapted to a dissipative circuit, and the reason it is cheap here rests on an observation that is easy
to get wrong — as indeed I did at first, when I estimated its cost at over four hundred qubits.

Introduce a **clock register** $C$ of $n_c=\lceil\log_2(n+1)\rceil$ qubits together with one **flag**
qubit, and prepare the clock in the uniform superposition
$$|c_0\rangle=\frac{1}{\sqrt{n+1}}\sum_{t=0}^{n}|t\rangle_C ,$$
which for $n=2^{n_c}-1$ is simply a Hadamard on each clock qubit. The propagation step is then made
*conditional on the clock*: at step $j\in\{1,\dots,n\}$ we apply
$$W_j=\sum_{t}|t\rangle\langle t|_C\otimes
\begin{cases} U_S & t\ge j\\[2pt] \mathbb 1 & t<j\end{cases}$$
to the clock together with the system and environment, and then reset the environment as before. In
practice $W_j$ is never built as one large matrix — that would make the gate span every qubit and the
simulation cost explode. Instead the predicate is computed into the flag qubit by the reversible oracle
$$F_j:\;|t\rangle_C|f\rangle\;\longmapsto\;|t\rangle_C\,\big|f\oplus[t\ge j]\big\rangle ,$$
the channel step is applied controlled on that single flag, and $F_j$ is applied a second time to
uncompute it, since $F_j^2=\mathbb 1$. Every gate then acts on at most $\max(n_c+1,\,1+n_{\rm sys}+n_{\rm env})$
qubits regardless of how long the trajectory is.

The effect is immediate: in the branch labelled $t$, the flag is set exactly for the steps $j\le t$, so
that branch receives exactly $t$ applications of the channel and none afterwards. Writing $\Phi$ for the
channel of §4, induction over $j$ gives for the diagonal blocks of the joint clock–system state
$$\big\langle t\big|\,\rho_{CS}^{(j)}\,\big|t\big\rangle=\frac{1}{n+1}\,\Phi^{\min(t,j)}(\rho_0),$$
and therefore, after all $n$ steps,
$$\boxed{\;\big\langle t\big|\,\rho_{CS}\,\big|t\big\rangle=\frac{1}{n+1}\,\Phi^{t}(\rho_0)=\frac{1}{n+1}\,\rho(t)\;}$$

Here is the point I initially missed. The environment reset does **not** preserve coherence between
clock branches: for $t\ge j$ the environment is entangled with the system while for $t<j$ it is still
$|0\rangle$, so tracing it out suppresses the off-diagonal blocks $|t\rangle\langle t'|$. I took this to
mean that the construction required the evolution to stay unitary, hence a *fresh* environment register
per step, hence $n\cdot n_{\rm env}+n_{\rm sys}+n_c\approx409$ qubits for $n=100$. That conclusion was
wrong, because **the clock is only ever measured in the computational basis**. Coherence between
different $t$ is never used; only the diagonal enters any measurement statistic, and the boxed identity
shows the diagonal is exactly right. One reused environment register therefore suffices, and the
overhead collapses from $\mathcal O(n)$ qubits to $\mathcal O(\log n)$.

Measuring clock and system together in the computational basis at the very end yields
$$P(t,i)=\frac{1}{n+1}\,\big[\rho(t)\big]_{ii},\qquad\text{hence}\qquad
P(i\mid t)=\big[\rho(t)\big]_{ii}=p_i(t),$$
the conditioning being exact because $\operatorname{Tr}\rho(t)=1$ for every $t$. Each shot returns one
pair $(t,i)$; sorting the shots by their clock value and normalising within each bin reconstructs the
population dynamics at **all** times from that one measurement.

### Complexity

The register cost is $n_{\rm sys}+n_{\rm env}+\lceil\log_2(n+1)\rceil+1$, i.e. the deterministic circuit
of §4 plus a logarithmic overhead; for the run below, $2+4+5+1=12$ qubits carry $32$ time points. The
gate count is $n$ controlled channel steps and $2n$ flag oracles, so $\mathcal O(n)$ gates of bounded
width and depth $\mathcal O(n)$ — the same asymptotics as the single-time circuit.

The shot cost deserves a precise statement, because it is the one place where nothing is gained. With
$S$ shots, roughly $S/(n+1)$ land in each clock bin, so resolving every $p_i(t)$ to accuracy $\varepsilon$
needs $S=\mathcal O\!\big(n/\varepsilon^{2}\big)$. Running $n$ separate circuits with
$\mathcal O(1/\varepsilon^{2})$ shots each costs exactly the same, and no scheme can do better: a single
shot carries $\mathcal O(1)$ bits about a single time, so $n$ time points require $\Omega(n)$ shots. The
clock therefore buys **one circuit and one measurement instead of $n$ circuits**, not a reduction in
sampling effort. If some times matter more than others, the uniform superposition can be replaced by
$\sum_t\sqrt{w_t}|t\rangle$, which reallocates shots as $P(t)\propto w_t$ at no extra cost.

### The non-Markovian case

The construction is agnostic to what the per-step circuit does, so route B inherits it verbatim: one
replaces $U_S$ by the dilated $U_H$ of §6 and controls it on the flag. Two things change. First, the
ancilla record must survive until the end, so the single reused ancilla of §5 becomes $n$ fresh ones —
the reason route B is demonstrated below on a shorter trajectory. Second, post-selecting all ancillas on
$|0\rangle$ keeps branch $t$ only with its own accumulated probability, so the clock marginal is no
longer flat but
$$P(t)\;\propto\;\prod_{j\le t}p_j=\frac{\|H_m^{\,t}y_0\|^2}{s^{2t}},$$
which decays with $t$ and simply means later times receive fewer shots. Crucially the *conditional*
distribution $P(i\mid t)$ is untouched by this weighting, so the reconstructed populations remain
unbiased — the measurement below reproduces the reference trajectory to $5.6\times10^{-16}$ while the
clock weight drops from $0.125$ to $0.061$ across eight steps.

---